In [ ]:
# Filename: process_legal.py
# pip install tiktoken sentencepiece protobuf
# ---------------------------------------------------------
# 1. CONFIGURATION
# 2. HELPERS
#   a. loading stopwords
#   b. loading Thesaurus
#   c. verify/validate quantization before model load
#   d. model load from previous model save, if fails continue from scratch
#   e. model unload (explicit RAM/VRAM release)
# 3. RAG MANAGER
# 4. PIPELINE  — three sequential phases, one model resident at a time
#   a. Phase 1 – Ingest: read all files, build RAG (sentence-transformer only)
#   b. Phase 2 – Summarise: CPU summariser → summarise all → unload
#   c. Phase 3 – Generate: CUDA generator → RAG-retrieve + generate → unload
# 5. MAIN FUNCTION
# ---------------------------------------------------------

import gc, os, re, torch
torch.set_num_threads(24)

USE_CUDA = torch.cuda.is_available()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "max_split_size_mb:128,"
    "garbage_collection_threshold:0.8"
)

if USE_CUDA:
    # if using Windows Native change to False
    torch.backends.cuda.enable_flash_sdp(False)
    # if using Windows Native change to False
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    # if using Windows Native change to True and uncomment it
    torch.backends.cuda.enable_math_sdp(True)

import pandas as pd
from datetime import datetime
from typing import List, Set

from langchain_community.document_loaders import WebBaseLoader, Docx2txtLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from striprtf.striprtf import rtf_to_text
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


# =============================================================================
# 1. CONFIGURATION
# =============================================================================
CONFIG = {
    "mode": "title",  # "keyword" or "title"
    #"main_model_id": "Qwen/Qwen3-4B-Thinking-2507",
    "main_model_id": "nvidia/Nemotron-Terminal-8B",
    "summ_model_id": "slovak-nlp/mistral-sk-7b",
    "emb_model_id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "input_dir": "./Input",
    "output_dir": "./Output",
    "models_dir": "./Models",
    "rag_dir": "./RAG_store",
    "thesaurus_path": "./Thesaurus/SK_Local_Thesaurus.xlsx",
    "thesaurus_col": "slovak_terms",
    "stopwords_path": "./Input/stopword_dictionary.txt",
    "allowed_domains": ["slov-lex.sk", "justice.gov.sk", "najpravo.sk", "zakonypreludi.sk"],
    "chunk_size": 1200,
    "chunk_overlap": 200,
    "top_k": 4,
}

for d in (CONFIG["output_dir"], CONFIG["rag_dir"], CONFIG["models_dir"]):
    os.makedirs(d, exist_ok=True)


# =============================================================================
# 2. HELPERS
# =============================================================================

def load_rtf(path: str) -> List[Document]:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return [Document(page_content=rtf_to_text(f.read()), metadata={"source": path})]
    except Exception as e:
        print(f"RTF parsing failed for {path}: {e}")
        return []

# a. loading stopwords
def load_stopwords(path: str) -> Set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}

# b. loading Thesaurus
def load_thesaurus(path: str, col: str, stopwords: Set[str]) -> List[str]:
    if not os.path.exists(path):
        return []
    df = pd.read_excel(path, engine="openpyxl")
    if col not in df.columns:
        return []
    seen, terms = set(), []
    for cell in df[col].dropna().astype(str):
        for t in re.split(r"[,\n;]+", cell):
            t = t.strip()
            if len(t) > 1 and t.lower() not in stopwords and t not in seen:
                seen.add(t)
                terms.append(t)
    print(f"[THESAURUS] Loaded {len(terms)} terms.")
    return terms

def match_terms(text: str, terms: List[str], limit: int = 10) -> str:
    if not text or not terms:
        return ""
    hits = [t for t in terms if re.search(rf"\b{re.escape(t)}\b", text, re.IGNORECASE)]
    hits.sort(key=len, reverse=True)
    return ", ".join(hits[:limit])

# c. verify/validate quantization before model load
def _verify_quantization(quantize: str | None, device: str) -> str | None:
    if quantize is None:
        return None

    if device == "cpu":
        print(f"Quantization '{quantize}' not supported on CPU → fallback to fp32.")
        return None

    if not torch.cuda.is_available():
        print("CUDA not available → fallback to fp32.")
        return None

    try:
        import bitsandbytes as bnb
        print(f"Bitsandbytes {bnb.__version__} detected.")
    except ImportError:
        print("Bitsandbytes not installed → fallback to fp16.")
        return None
    except Exception as e:
        print(f"Bitsandbytes import failed: {e} → fallback to fp16.")
        return None

    try:
        if quantize == "4bit":
            _ = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
        elif quantize == "8bit":
            _ = BitsAndBytesConfig(load_in_8bit=True)
        else:
            print(f"Unknown quantization mode '{quantize}' → fallback to fp16.")
            return None
    except Exception as e:
        print(f"BitsAndBytesConfig init failed: {e} → fallback to fp16.")
        return None

    try:
        free_mem, _ = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024 ** 3)
        print(f"CUDA free VRAM: {free_gb:.2f} GB")
        if free_gb < 2.0:
            print(f"Very low VRAM ({free_gb:.2f} GB) — quantization may fail at load.")
    except Exception:
        pass

    print(f"Verification OK for '{quantize}'.")
    return quantize

# d. model load from previous model save, if fails continue from scratch
def load_model(model_id: str, device: str, quantize: str | None = None):
    quantize = _verify_quantization(quantize, device)

    if quantize in ("4bit", "8bit"):
        variant = quantize
    elif device.startswith("cuda"):
        variant = "fp16"
    else:
        variant = "fp32"

    safe_name  = model_id.replace("/", "_")
    local_path = os.path.join(CONFIG["models_dir"], f"{safe_name}__{variant}")

    has_config  = os.path.exists(os.path.join(local_path, "config.json"))
    weight_files = (
                    "model.safetensors",
                    "pytorch_model.bin",
                    "model.safetensors.index.json",
                    "pytorch_model.bin.index.json",
                    )
    has_weights = any(os.path.exists(os.path.join(local_path, w)) for w in weight_files)
    from_local  = has_config and has_weights

    if from_local:
        source = local_path
        print(f"[{device.upper()}] Loading {model_id} from LOCAL cache: {local_path} (variant={variant})")
    else:
        source = model_id
        print(f"[{device.upper()}] Loading {model_id} from HuggingFace (variant={variant})")

    bnb_config = None
    if quantize == "4bit":
        bnb_config = BitsAndBytesConfig(
                                        load_in_4bit=True,
                                        bnb_4bit_quant_type="nf4",
                                        bnb_4bit_compute_dtype=torch.float16,
                                        bnb_4bit_use_double_quant=True,
                                        )
    elif quantize == "8bit":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    kwargs = dict(device_map=device, trust_remote_code=True, low_cpu_mem_usage=True)

    if from_local:
        if bnb_config is None:
            kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32
    else:
        if bnb_config is not None:
            kwargs["quantization_config"] = bnb_config
        else:
            kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32

    try:
        tok = AutoTokenizer.from_pretrained(source, trust_remote_code=True)

        try:
            model = AutoModelForCausalLM.from_pretrained(source, attn_implementation="sdpa", **kwargs)
        except Exception as e:
            if bnb_config is not None:
                print(f"[QUANT] Quantized load failed: {e}")
                print("[QUANT] Retrying without quantization (fp16) …")
                kwargs.pop("quantization_config", None)
                kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32
                model = AutoModelForCausalLM.from_pretrained(source, **kwargs)
                bnb_config = None
                variant    = "fp16"
                local_path = os.path.join(CONFIG["models_dir"], f"{safe_name}__{variant}")
            else:
                raise

        if not from_local:
            try:
                os.makedirs(local_path, exist_ok=True)
                tok.save_pretrained(local_path)
                model.save_pretrained(local_path, safe_serialization=True)
                print(f"[Model ({variant})] cached to {local_path}")
            except Exception as e:
                print(f"Could not save model locally ({variant}): {e}")
                try:
                    tok.save_pretrained(local_path)
                    print(f"Tokenizer-only cached to {local_path}")
                except Exception as e2:
                    print(f"Could not save tokenizer: {e2}")

        return pipeline("text-generation", model=model, tokenizer=tok)

    except Exception as e:
        print(f"Load failed ({device}, {quantize}): {e}")
        return None

def run_llm(pipe, prompt: str, split_on: str) -> str:
    try:
        res = pipe(prompt)
        return res[0]["generated_text"].split(split_on)[-1].strip()
    except Exception as e:
        print(f"LLM call failed: {e}")
        return ""

# e. model unload — explicit RAM / VRAM release
def unload_model(pipe) -> None:
    if pipe is None:
        return
    try:
        del pipe.model
        del pipe.tokenizer
    except AttributeError:
        pass
    del pipe
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()
    print("[MEM] Model unloaded.")

# =============================================================================
# 3. RAG MANAGER
# =============================================================================
class RAGManager:
    def __init__(self):
        print("Loading embeddings on CPU …")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=CONFIG["emb_model_id"],
            model_kwargs={"device": "cpu"},
        )
        self.vs = Chroma(collection_name="sk_legal_docs",persist_directory=CONFIG["rag_dir"],embedding_function=self.embeddings)
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=CONFIG["chunk_size"],chunk_overlap=CONFIG["chunk_overlap"])

    def add_text(self, text: str, metadata: dict):
        docs = self.splitter.create_documents([text], metadatas=[metadata])
        if docs:
            self.vs.add_documents(docs)

    def search(self, query: str) -> str:
        docs = self.vs.similarity_search(query, k=CONFIG["top_k"])
        return "\n\n".join(d.page_content for d in docs)

    def ingest_web(self, urls: List[str]):
        valid = [u for u in urls if any(d in u for d in CONFIG["allowed_domains"])]
        if not valid:
            return
        try:
            docs = self.splitter.split_documents(WebBaseLoader(valid).load())
            self.vs.add_documents(docs)
            print(f"[WEB] Ingested {len(docs)} chunks.")
        except Exception as e:
            print(f"[WEB] Ingest failed: {e}")

# =============================================================================
# 4. PIPELINE — three sequential phases
# =============================================================================

LOADERS = {".docx": Docx2txtLoader,".rtf":  load_rtf,".txt":  lambda p: TextLoader(p, encoding="utf-8"),}

def read_file(path: str) -> str:
    ext    = os.path.splitext(path)[1].lower()
    loader = LOADERS.get(ext)
    if loader is None:
        return ""
    try:
        docs = loader(path).load() if ext != ".rtf" else loader(path)
        return "\n".join(d.page_content for d in docs)
    except Exception as e:
        print(f"File load failed: {e}")
        return ""

# a. Phase 1 – Ingest: read all files, build RAG (sentence-transformer only)
def phase_ingest(files: List[str], rag: RAGManager) -> dict:
    corpus = {}
    for filepath in files:
        filename = os.path.basename(filepath)
        print(f"[INGEST] {filename}")
        text = read_file(filepath)
        if not text.strip():
            print(f"  → empty, skipping.")
            continue
        rag.add_text(text, {"file": filename})
        corpus[filepath] = text
    print(f"[INGEST] Done — {len(corpus)} files indexed.\n")
    return corpus

# b. Phase 2 – Summarise: CPU summariser → summarise all files → caller unloads
def phase_summarise(corpus: dict, summarizer) -> dict:
    summaries = {}
    for filepath, text in corpus.items():
        filename = os.path.basename(filepath)
        print(f"[SUMM] {filename}")
        prompt = (
                    f"Zhrň nasledovný text do stručného odstavca (max 5 viet). Píš po slovensky.\n\n"
                    f"TEXT:\n{text[:12000]}\n\nZHRNUTIE:"
                )
        summary = run_llm(summarizer, prompt, "ZHRNUTIE:") or text[:500]
        summaries[filepath] = summary
    print(f"[SUMM] Done — {len(summaries)} summaries produced.\n")
    return summaries

# c. Phase 3 – Generate: CUDA generator → RAG-retrieve + generate → caller unloads
def phase_generate(
    corpus: dict,
    summaries: dict,
    rag: RAGManager,
    generator,
    terms: List[str],
) -> List[dict]:
    mode = CONFIG["mode"]
    prompt_instruction = (
        "Navrhni JEDEN presný právny nadpis (Title)"
        if mode == "title"
        else "Vyber JEDEN kľúčový pojem (Keyword)"
    )
    suffix = "Nadpis:" if mode == "title" else "Kľúčový pojem:"

    results = []
    for filepath, text in corpus.items():
        filename = os.path.basename(filepath)
        print(f"[GEN] {filename}")
        summary = summaries.get(filepath, text[:500])
        matched = match_terms(text, terms)
        ctx     = rag.search(f"{summary} {matched}")

        gen_prompt = (
            f"Kontext:\n{ctx[:2000]}\n\n"
            f"Zadanie: Na základe súhrnu a pojmov ({matched}) {prompt_instruction}.\n\n"
            f"Súhrn:\n{summary}\n\n{suffix}"
        )
        raw    = run_llm(generator, gen_prompt, suffix)
        output = raw.split("\n")[0].strip('" ') if raw else "ERROR"

        results.append({
                        "file":             filename,
                        "mode":             mode,
                        "summary":          summary,
                        "priority_terms":   matched,
                        "generated_output": output,
                        })

    print(f"[GEN] Done — {len(results)} outputs generated.\n")
    return results

# =============================================================================
# 5. MAIN FUNCTION
# =============================================================================
def main():
    if USE_CUDA:
        torch.cuda.empty_cache()
    print(f"STARTING PIPELINE (Mode: {CONFIG['mode']})")
    print("Hardware: RTX 4070 Super (optimising for 12 GB VRAM)\n")

    stopwords = load_stopwords(CONFIG["stopwords_path"])
    terms     = load_thesaurus(CONFIG["thesaurus_path"], CONFIG["thesaurus_col"], stopwords)

    files = [
        os.path.join(CONFIG["input_dir"], f)
        for f in os.listdir(CONFIG["input_dir"])
        if f.lower().endswith((".docx", ".rtf", ".txt"))
    ]

    # ── Phase 1: sentence-transformer only (CPU, ~400 MB RAM) ─────────────
    print("=" * 60)
    print("PHASE 1 — INGEST (sentence-transformer)")
    print("=" * 60)
    rag = RAGManager()                          # sentence-transformer loads here
    rag.ingest_web(["https://www.slov-lex.sk/pravne-predpisy/SK/ZZ/2005/300/"])
    corpus = phase_ingest(files, rag)

    if not corpus:
        print("No files to process. Exiting.")
        return

    # ── Phase 2: CPU summariser (~28 GB RAM for 7B fp32) ──────────────────
    print("=" * 60)
    print("PHASE 2 — SUMMARISE (CPU model)")
    print("=" * 60)
    summarizer = load_model(CONFIG["summ_model_id"], "cpu")
    if not summarizer:
        print("Summariser failed to load. Exiting.")
        return

    summaries = phase_summarise(corpus, summarizer)

    # Free CPU RAM before asking CUDA to load the generator
    unload_model(summarizer)
    print("[MEM] Summariser released — RAM freed before GPU load.\n")

    # ── Phase 3: quantised CUDA generator (~2–4 GB VRAM for 2B 8-bit) ─────
    print("=" * 60)
    print("PHASE 3 — GENERATE (CUDA model)")
    print("=" * 60)
    generator = load_model(CONFIG["main_model_id"], "cuda:0", quantize="8bit")
    if not generator:
        print("Generator failed to load. Exiting.")
        return

    results = phase_generate(corpus, summaries, rag, generator, terms)

    unload_model(generator)
    print("[MEM] Generator released.\n")

    # ── Save results ───────────────────────────────────────────────────────
    if results:
        df       = pd.DataFrame(results)
        out_name = f"output_{CONFIG['mode']}_{datetime.now():%Y-%m-%d}.csv"
        df.to_csv(
            os.path.join(CONFIG["output_dir"], out_name),
            index=False,
            encoding="utf-8-sig",
        )
        print(f"Saved → {out_name}")
        print(df[["file", "generated_output"]].head())

    if USE_CUDA:
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

USER_AGENT environment variable not set, consider setting it to identify your requests.


STARTING PIPELINE (Mode: title)
Hardware: RTX 4070 Super (optimising for 12 GB VRAM)

[THESAURUS] Loaded 3000 terms.
PHASE 1 — INGEST (sentence-transformer)
Loading embeddings on CPU …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[WEB] Ingested 1 chunks.
[INGEST] 117694.docx
[INGEST] 117695.docx
[INGEST] 117696.docx
[INGEST] 117697.docx
[INGEST] 117698.docx
[INGEST] 117699.docx
[INGEST] 117700.docx
[INGEST] 117701.docx
[INGEST] 117702.docx
[INGEST] 117703.docx
[INGEST] stopword_dictionary.txt
[INGEST] Done — 11 files indexed.

PHASE 2 — SUMMARISE (CPU model)
[CPU] Loading slovak-nlp/mistral-sk-7b from HuggingFace (variant=fp32)


...

[GEN] 117702.docx


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[GEN] 117703.docx


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[GEN] stopword_dictionary.txt
[GEN] Done — 11 outputs generated.

[MEM] Model unloaded.
[MEM] Generator released.

Saved → output_title_2026-04-24.csv
          file                                   generated_output
0  117694.docx                JEDEN presný právny nadpis (Title).
1  117695.docx  **Delenie dôkazného bremena v civilnom súdnom ...
2  117696.docx  Pokiaľ ide o to, ktorú z týchto funkcií plní u...
3  117697.docx                                            <think>
4  117698.docx                               Odstúpenie od zmluvy
